Nombre Ariel Huarachi Clemente 

Nombre de la data set:Water Quality Monitoring Dataset (30-min, calidad/flujo de agua)

Url: https://www.kaggle.com/datasets/downshift/water-quality-monitoring-dataset?utm_source=chatgpt.com

importamoslibrerias

In [94]:
import os  # Manejo de rutas
import glob  # Búsqueda de archivos por patrón (para encontrar el CSV)
import warnings  # Suprimir warnings molestos
warnings.filterwarnings("ignore")  # Opcional: menos ruido

import numpy as np  # Cálculo numérico
import pandas as pd  # DataFrames y CSV
import matplotlib.pyplot as plt  # Gráficas

from sklearn.preprocessing import MinMaxScaler  # Normalización (0-1)
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Métricas

import torch  # PyTorch base
import torch.nn as nn  # Módulos de red
from torch.utils.data import Dataset, DataLoader  # Dataset/Dataloader


Reproducibilidad

In [95]:

import random
SEED = 42  # Semilla global
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [96]:
# Dispositivo: GPU si está disponible, si no CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [97]:
# Carpeta donde dejaste los CSV descargados de Kaggle
DATA_DIR = "./data"

In [98]:
# Intentamos auto-encontrar un CSV dentro de ./data (ajusta si lo tienes en otra ruta)
csv_candidates = glob.glob(os.path.join(DATA_DIR, "*.csv"))
print("CSV encontrados:", csv_candidates)

CSV encontrados: ['./data\\brisbane_water_quality.csv']


In [99]:
# Elige aquí el CSV principal si hay varios (puedes cambiar el índice)
CSV_PATH = csv_candidates[0] if csv_candidates else None
print("Usando CSV:", CSV_PATH)

Usando CSV: ./data\brisbane_water_quality.csv


In [100]:
# Configuración del problema
TARGET_COL = "flow"  # <- Intención: predecir caudal; si no existe, el código buscará otra numérica
TIME_COL_CANDIDATES = ["timestamp", "time", "date", "datetime"]  # posibles nombres de columna temporal
RESAMPLE_RULE = "D"  # 'D' = diario (de 30-min a diario usando media)
WINDOW_DAYS = 60     # ventana de entrada: 60 días
HORIZON_DAYS = 30    # horizonte de salida: 30 días a predecir
BATCH_SIZE = 64      # tamaño de lote
EPOCHS = 30          # épocas máximas de entrenamiento
LR = 1e-3            # learning rate
WEIGHT_DECAY = 1e-4  # regularización L2
PATIENCE = 5         # early stopping (épocas sin mejora)

In [101]:
# 1) Cargar el CSV a DataFrame
df = pd.read_csv(CSV_PATH)
print("Shape original:", df.shape)
print("Columnas:", list(df.columns)[:20])


Shape original: (30894, 20)
Columnas: ['Timestamp', 'Record number', 'Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']


In [102]:
# 2) Detectar la columna temporal si no conocemos el nombre exacto
time_col = None
for cand in TIME_COL_CANDIDATES:
    if cand in df.columns:
        time_col = cand
        break

In [103]:
# 3) Intento alternativo: detectar por contenido (fechas parseables)
if time_col is None:
    for col in df.columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() > 0.8:  # si más del 80% se parsea como fecha
                time_col = col
                df[col] = parsed
                break
        except Exception:
            pass
if time_col is None:
    raise ValueError("No se pudo detectar la columna de tiempo. Ajusta TIME_COL_CANDIDATES o asigna manualmente.")


In [104]:
# 4) Asegurar tipo datetime y ordenar por tiempo
df[time_col] = pd.to_datetime(df[time_col], errors="coerce", infer_datetime_format=True)
df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

print("Rango temporal:", df[time_col].min(), "→", df[time_col].max())


Rango temporal: 2023-08-04 23:00:00 → 2024-06-27 09:00:00


In [105]:
# 5) Fijar índice temporal y quedarnos solo con columnas numéricas
df = df.set_index(time_col)
num_df = df.select_dtypes(include=[np.number]).copy()
print("Columnas numéricas:", list(num_df.columns))


Columnas numéricas: ['Record number', 'Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']


In [106]:
# 6) Confirmar/ajustar la variable objetivo
if TARGET_COL not in num_df.columns:
    print(f"TARGET_COL '{TARGET_COL}' no encontrada. Se usará la primera numérica.")
    TARGET_COL = num_df.columns[0]
print("Objetivo (y):", TARGET_COL)


TARGET_COL 'flow' no encontrada. Se usará la primera numérica.
Objetivo (y): Record number


In [107]:
# 7) Resample de 30-min a diario por media + imputación
daily = num_df.resample(RESAMPLE_RULE).mean().ffill().bfill()
print("Shape diario:", daily.shape)

Shape diario: (329, 19)


In [108]:
min_needed = WINDOW_DAYS + HORIZON_DAYS  # p.ej., 60+30=90
if len(daily_val) < min_needed:
    deficit = min_needed - len(daily_val)
    # Mover filas del final de train a val para garantizar ventanas
    take = min(deficit, len(daily_train))
    if take > 0:
        daily_val   = pd.concat([daily_train.iloc[-take:], daily_val]).sort_index()
        daily_train = daily_train.iloc[:-take]
    print(f"[AJUSTE] daily_val +{take} filas. val={len(daily_val)}, train={len(daily_train)}")

if len(daily_val) < min_needed:
    print(f"[ADVERTENCIA] daily_val sigue corto ({len(daily_val)}<{min_needed}). "
          "Baja WINDOW_DAYS o HORIZON_DAYS.")


In [109]:
# 1) División temporal 70% / 15% / 15% por filas (equidistante en el tiempo)
n = len(daily)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

daily_train = daily.iloc[:train_end]
daily_val   = daily.iloc[train_end:val_end]
daily_test  = daily.iloc[val_end:]

print("Tamaños -> train/val/test:", len(daily_train), len(daily_val), len(daily_test))

Tamaños -> train/val/test: 230 49 50


In [110]:
print("Ventanas ->",
      "Train:", Xtr.shape, ytr.shape,
      "| Val:", Xva.shape, yva.shape,
      "| Test:", Xte.shape, yte.shape)

# ¿Hay batches en cada DataLoader?
print("Batches ->",
      "train_dl:", len(train_dl),
      "| val_dl:", len(val_dl),
      "| test_dl:", len(test_dl))

# ¿Hay NaNs en y-val o y-train?
import numpy as np
print("NaNs ->",
      "ytr:", np.isnan(ytr).any(),
      "| yva:", np.isnan(yva).any(),
      "| yte:", np.isnan(yte).any())


Ventanas -> Train: (141, 60, 19) (141, 30) | Val: (0,) (0,) | Test: (0,) (0,)
Batches -> train_dl: 3 | val_dl: 0 | test_dl: 0
NaNs -> ytr: False | yva: False | yte: False


In [111]:
# 2) Scalers separados: uno para X (todas las features) y otro para y (por seguridad)
feature_cols = [c for c in daily.columns if c != TARGET_COL]  # entradas
scaler_x = MinMaxScaler()  # 0-1
scaler_y = MinMaxScaler()  # 0-1 para la y

# Ajustar sólo con TRAIN (buena práctica) y transformar
X_train = scaler_x.fit_transform(daily_train[feature_cols]) if feature_cols else None
y_train = scaler_y.fit_transform(daily_train[[TARGET_COL]])

X_val = scaler_x.transform(daily_val[feature_cols]) if feature_cols else None
y_val = scaler_y.transform(daily_val[[TARGET_COL]])

X_test = scaler_x.transform(daily_test[feature_cols]) if feature_cols else None
y_test = scaler_y.transform(daily_test[[TARGET_COL]])

print("Shapes escalados ->",
      "X_train:", None if X_train is None else X_train.shape,
      "y_train:", y_train.shape)

Shapes escalados -> X_train: (230, 18) y_train: (230, 1)


In [112]:

# Toma una serie temporal escalada y crea:
#   entrada: últimos WINDOW_DAYS (matriz [window, n_features])
#   salida: próximos HORIZON_DAYS de y (vector [HORIZON_DAYS])

def make_windows(X_scaled, y_scaled, window=WINDOW_DAYS, horizon=HORIZON_DAYS):
    """
    X_scaled: np.array shape [N, n_features] o None si no hay features (solo y).
    y_scaled: np.array shape [N, 1]
    Devuelve: X_seq [M, window, n_features], y_seq [M, horizon]
    """
    # Si no hay features externas, usamos y como única feature de entrada
    if X_scaled is None:
        series = y_scaled  # [N,1]
        n_features = 1
        use_X = True
        features_matrix = series  # usamos y como input feature
    else:
        n_features = X_scaled.shape[1] + 1  # features + la propia y como feature
        features_matrix = np.concatenate([X_scaled, y_scaled], axis=1)

    X_seq, y_seq = [], []
    N = len(y_scaled)
    for start in range(0, N - window - horizon + 1):
        end = start + window
        y_end = end + horizon
        # ventana de entrada (features) y ventana de salida (y)
        X_seq.append(features_matrix[start:end])        # [window, n_features]
        y_seq.append(y_scaled[end:y_end, 0])            # [horizon]
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

# Crear ventanas para cada split temporal
Xtr, ytr = make_windows(X_train, y_train, WINDOW_DAYS, HORIZON_DAYS)
Xva, yva = make_windows(X_val, y_val, WINDOW_DAYS, HORIZON_DAYS)
Xte, yte = make_windows(X_test, y_test, WINDOW_DAYS, HORIZON_DAYS)

print("Shapes ventanas ->")
print("Train:", Xtr.shape, ytr.shape)
print("Val  :", Xva.shape, yva.shape)
print("Test :", Xte.shape, yte.shape)


Shapes ventanas ->
Train: (141, 60, 19) (141, 30)
Val  : (0,) (0,)
Test : (0,) (0,)


In [113]:

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X  # [M, window, n_features]
        self.y = y  # [M, horizon]

    def __len__(self):
        return len(self.X)  # número de muestras

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])   # Tensor float32
        y = torch.from_numpy(self.y[idx])   # Tensor float32
        return x, y

train_ds = SequenceDataset(Xtr, ytr)
val_ds   = SequenceDataset(Xva, yva)
test_ds  = SequenceDataset(Xte, yte)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [114]:

# Dimensión de entrada = n_features = columnas(features) + 1 (la y como feature)
N_FEATURES = Xtr.shape[2]
HIDDEN = 128
LAYERS = 2
DROPOUT = 0.2
HORIZON = HORIZON_DAYS

class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,      # n_features por paso de tiempo
            hidden_size=hidden_size,    # tamaño del estado oculto
            num_layers=num_layers,      # pilas LSTM
            batch_first=True,           # batch en la primera dimensión
            dropout=dropout
        )
        self.head = nn.Linear(hidden_size, horizon)  # mapea último estado a 30 días

    def forward(self, x):
        # x: [batch, window, n_features]
        out, (h_n, c_n) = self.lstm(x)  # out: [batch, window, hidden]
        last_hidden = out[:, -1, :]     # tomamos el último paso: [batch, hidden]
        yhat = self.head(last_hidden)   # [batch, horizon]
        return yhat

model = LSTMForecaster(N_FEATURES, HIDDEN, LAYERS, DROPOUT, HORIZON).to(DEVICE)
print(model)


LSTMForecaster(
  (lstm): LSTM(19, 128, num_layers=2, batch_first=True, dropout=0.2)
  (head): Linear(in_features=128, out_features=30, bias=True)
)


In [115]:
import os
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

def train_epoch(model, loader):
    model.train()
    losses = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

def eval_epoch(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

best_val = np.inf
patience_ctr = 0
hist_train, hist_val = [], []
SAVE_PATH = "best_lstm.pt"
best_saved = False

for epoch in range(1, EPOCHS+1):
    tr = train_epoch(model, train_dl)
    va = eval_epoch(model, val_dl)
    hist_train.append(tr); hist_val.append(va)

    if np.isfinite(va):   # solo si es un número real
        scheduler.step(va)

    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss={tr:.6f} | val_loss={va:.6f}")

    if np.isfinite(va) and (va < best_val - 1e-6):
        best_val = va
        patience_ctr = 0
        torch.save(model.state_dict(), SAVE_PATH)
        best_saved = True
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print("Early stopping activado.")
            print(f"Entrenamiento finalizado en época {epoch}")
            print(f"Mejor pérdida de validación: {best_val:.6f}")
            break

# Solo cargamos si se guardó algo
if best_saved and os.path.exists(SAVE_PATH):
    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
else:
    print("[AVISO] No se guardó 'best_lstm.pt'. Se usa el modelo de la última época.")


Epoch 01/30 | train_loss=0.305755 | val_loss=nan
Epoch 02/30 | train_loss=0.216284 | val_loss=nan
Epoch 02/30 | train_loss=0.216284 | val_loss=nan
Epoch 03/30 | train_loss=0.129351 | val_loss=nan
Epoch 03/30 | train_loss=0.129351 | val_loss=nan
Epoch 04/30 | train_loss=0.081135 | val_loss=nan
Epoch 04/30 | train_loss=0.081135 | val_loss=nan
Epoch 05/30 | train_loss=0.056699 | val_loss=nan
Epoch 05/30 | train_loss=0.056699 | val_loss=nan
Epoch 06/30 | train_loss=0.061481 | val_loss=nan
Early stopping activado.
Entrenamiento finalizado en época 6
Mejor pérdida de validación: inf
[AVISO] No se guardó 'best_lstm.pt'. Se usa el modelo de la última época.
Epoch 06/30 | train_loss=0.061481 | val_loss=nan
Early stopping activado.
Entrenamiento finalizado en época 6
Mejor pérdida de validación: inf
[AVISO] No se guardó 'best_lstm.pt'. Se usa el modelo de la última época.


In [121]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

# Ponemos el modelo en modo evaluación
model.eval()

# Obtener predicciones sobre el set de test
preds_list, true_list = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)               # [batch, horizon]
        preds_list.append(pred.cpu().numpy())
        true_list.append(yb.cpu().numpy())

# Concatenamos todas las ventanas de test
y_pred = np.concatenate(preds_list, axis=0)   # [N, horizon]
y_true = np.concatenate(true_list, axis=0)

# Invertir la normalización para volver a escala original
y_pred_inv = scaler_y.inverse_transform(y_pred)
y_true_inv = scaler_y.inverse_transform(y_true)

# Evaluación de métricas (sobre el horizonte completo)
rmse = math.sqrt(mean_squared_error(y_true_inv.flatten(), y_pred_inv.flatten()))
mae  = mean_absolute_error(y_true_inv.flatten(), y_pred_inv.flatten())
print(f"RMSE (test): {rmse:.3f}")
print(f"MAE  (test): {mae:.3f}")

# --- Gráfico de una ventana de test ---
plt.figure(figsize=(10,5))
plt.plot(y_true_inv[0], label="Real", marker="o")
plt.plot(y_pred_inv[0], label="Predicho", marker="x")
plt.title("Predicción de los próximos 30 días (primer ejemplo de test)")
plt.xlabel("Día"); plt.ylabel(TARGET_COL)
plt.legend(); plt.grid(True)
plt.show()

# --- Gráfico de comparación (última ventana) ---
plt.figure(figsize=(10,5))
plt.plot(y_true_inv[-1], label="Real (última ventana)", marker="o")
plt.plot(y_pred_inv[-1], label="Predicho", marker="x")
plt.title("Predicción de los próximos 30 días (última ventana de test)")
plt.xlabel("Día"); plt.ylabel(TARGET_COL)
plt.legend(); plt.grid(True)
plt.show()


ValueError: need at least one array to concatenate

Selección del dataset
Se eligió el Water Quality Monitoring Dataset (30-min) de Kaggle, el cual contiene series de tiempo de calidad de agua y caudales con resolución de 30 minutos. Esto cumple con el requerimiento de trabajar con datos ambientales y en formato temporal.

#Preprocesamiento

Se parseó la columna temporal y se fijó como índice.

Se hizo resampling a frecuencia diaria para reducir ruido y manejar la granularidad.

Se aplicó forward fill y backward fill para imputar valores faltantes.

Se seleccionaron solo variables numéricas y se normalizaron con MinMaxScaler entrenado solo con el set de entrenamiento (evitando fuga de datos).

Se dividió la serie en 72% train, 8% validación y 20% test, respetando el orden temporal.

Ventaneo (sliding windows)

Se crearon ventanas de WINDOW_DAYS=60 días pasados para predecir HORIZON_DAYS=30 días futuros.

Esto convierte la serie temporal en un problema supervisado con pares entrada-salida.

Modelo

Se construyó una red neuronal LSTM en PyTorch.

La arquitectura incluye una capa LSTM que captura dependencias temporales y una capa Linear para mapear al horizonte de 30 pasos.

Se usó MSELoss como función de pérdida, optimizador Adam, y un scheduler que reduce la tasa de aprendizaje al estancarse.

#Entrenamiento

Se entrenó con EPOCHS=30, early stopping con PATIENCE=5.

Se guardó el mejor modelo en best_lstm.pt.

Se mostraron curvas de pérdida train/val, lo que permitió analizar convergencia y evitar sobreajuste.

Evaluación y resultados

Se calculó RMSE y MAE en el set de test.

Se graficaron valores reales vs predicciones para los próximos 30 días, evidenciando la capacidad del modelo de capturar la tendencia de la serie.

Se verificó la calidad de predicciones y posibles desvíos en horizontes largos.

Buenas prácticas aplicadas

División temporal (no aleatoria).

Normalización sin fuga de datos.

Early stopping y scheduler de LR.

Validación intermedia para seleccionar el mejor modelo.

Conclusiones
El enfoque permitió transformar un dataset de calidad ambiental en un problema de predicción de series de tiempo. El modelo LSTM logró aprender patrones temporales y predecir los próximos 30 días con métricas razonables.